# Meaning as direction in embedding space
1. GloVe 6B, 100-d word vectors (Pennington et al. 2014): only the 100-d file is read out of the official zip by HTTP byte ranges (about 134 MB, once; cached in ~/.cache/campusx/glove/). We keep the 100,000 most frequent words.
2. Nearest neighbours of "tower".
3. Differences between related words point the same way: cosine of (b - a) with (woman - man), against random word pairs.
4. Vector arithmetic: king - man + woman and other analogies, with honest ranks.
5. A dot-product probe: the plural direction, on held-out nouns and on one ... ten.
6. The same tests on GPT-2 small's token-embedding table (`tools/gpt2_small.py`).
7. Coordinates for the animation (a 2-D plane through man, woman and king).

In [ ]:
import io, shutil, sys, time, urllib.error, urllib.request, zipfile
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT.parent / "tools"))
import gpt2_small as g
np.set_printoptions(precision=3, suppress=True)

GLOVE_URL = "https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip"   # linked from nlp.stanford.edu/projects/glove
GLOVE = Path.home() / ".cache" / "campusx" / "glove" / "glove.6B.100d.txt"


class HTTPFile(io.RawIOBase):
    """A read-only file over HTTP byte ranges, so `zipfile` can pull one member out of the 822 MB zip."""
    def __init__(self, url):
        self.url, self.pos = url, 0
        head = urllib.request.Request(url, method="HEAD", headers={"User-Agent": "curl/8.5.0"})
        self.size = int(urllib.request.urlopen(head).headers["Content-Length"])
    def readable(self): return True
    def seekable(self): return True
    def tell(self): return self.pos
    def seek(self, off, whence=0):
        self.pos = [off, self.pos + off, self.size + off][whence]
        return self.pos
    def readinto(self, b):
        n = min(len(b), self.size - self.pos)
        if n <= 0:
            return 0
        req = urllib.request.Request(self.url, headers={"Range": f"bytes={self.pos}-{self.pos + n - 1}",
                                                        "User-Agent": "curl/8.5.0"})
        for k in range(8):                       # the server sometimes answers 503: wait and retry
            try:
                d = urllib.request.urlopen(req, timeout=120).read()
                break
            except urllib.error.HTTPError:
                time.sleep(2 ** k)
        b[:len(d)] = d
        self.pos += len(d)
        return len(d)


if not GLOVE.exists():
    GLOVE.parent.mkdir(parents=True, exist_ok=True)
    z = zipfile.ZipFile(io.BufferedReader(HTTPFile(GLOVE_URL), buffer_size=1 << 24))
    with z.open("glove.6B.100d.txt") as src, open(GLOVE, "wb") as dst:
        shutil.copyfileobj(src, dst, 1 << 24)

words, vecs = [], []
with open(GLOVE, encoding="utf8") as f:
    for i, line in enumerate(f):
        if i == 100_000:
            break
        p = line.rstrip().split(" ")
        words.append(p[0])
        vecs.append(np.array(p[1:], dtype=np.float32))
E = np.stack(vecs)
idx = {w: i for i, w in enumerate(words)}
U = E / np.linalg.norm(E, axis=1, keepdims=True)
print(E.shape, "first words (the file runs from most to least frequent):", words[:12])

## Nearest neighbours by cosine similarity

In [ ]:
def nearest(v, exclude=(), k=8, words=words, U=U):
    s = U @ (v / np.linalg.norm(v))
    out = []
    for i in np.argsort(-s):
        if words[i] not in exclude:
            out.append((words[i], round(float(s[i]), 3)))
        if len(out) == k:
            return out

nb_tower = nearest(E[idx["tower"]], exclude={"tower"})
print(nb_tower)
pd.DataFrame(nb_tower, columns=["word", "cosine"]).to_csv(DATA / "tower_neighbours.csv", index=False)

## Differences between related words point the same way
Cosine between each pair's difference (female word minus male word) and woman - man. Baseline: 5,000 random pairs of words drawn from the 10,000 most frequent.

In [ ]:
PAIRS = [("king", "queen"), ("uncle", "aunt"), ("nephew", "niece"), ("father", "mother"), ("son", "daughter"),
         ("brother", "sister"), ("boy", "girl"), ("husband", "wife"), ("prince", "princess"),
         ("grandfather", "grandmother"), ("he", "she"), ("actor", "actress")]


def pair_alignment(vec, ref=("man", "woman"), seed=0, vocab_ids=None):
    gdir = vec(ref[1]) - vec(ref[0])
    cos = lambda a, b: float(a @ b / np.linalg.norm(a) / np.linalg.norm(b))
    rows = [(f"{a} → {b}", cos(vec(b) - vec(a), gdir)) for a, b in PAIRS]
    rng = np.random.default_rng(seed)
    i, j = rng.choice(vocab_ids, (2, 5000))
    keep = i != j
    base = np.array([cos(M[b] - M[a], gdir) for a, b in zip(i[keep], j[keep])])
    return rows, base

M = E
rows_glove, base_glove = pair_alignment(lambda w: E[idx[w]], vocab_ids=np.arange(10_000))
for r in rows_glove:
    print(f"{r[0]:26s} {r[1]:.2f}")
print("random pairs: mean %.3f, 99th percentile of |cos| %.3f" % (base_glove.mean(), np.percentile(abs(base_glove), 99)))

## Vector arithmetic: king - man + woman
The answer is the word whose vector has the largest cosine with b - a + c (Pennington et al. 2014, §4.1). We report the ranking over all words and the ranking with the three question words removed.

In [ ]:
QUESTIONS = [("man", "woman", "king", "queen"), ("man", "woman", "uncle", "aunt"), ("man", "woman", "nephew", "niece"),
             ("man", "woman", "father", "mother"), ("man", "woman", "son", "daughter"), ("man", "woman", "boy", "girl"),
             ("man", "woman", "husband", "wife"), ("man", "woman", "prince", "princess"),
             ("man", "woman", "brother", "sister"), ("germany", "italy", "hitler", "mussolini"),
             ("japan", "germany", "sushi", "bratwurst")]


def analogy(vec, U, words, a, b, c, target, fmt=lambda w: w):
    v = vec(c) + vec(b) - vec(a)
    s = U @ (v / np.linalg.norm(v))
    order = np.argsort(-s)
    ranked = [words[i] for i in order[:2000]]
    excl = [w for w in ranked if w not in {fmt(a), fmt(b), fmt(c)}]
    rank = excl.index(fmt(target)) + 1 if fmt(target) in excl else ">2000"
    t = words.index(fmt(target)) if isinstance(words, list) else None
    return dict(question=f"{c} − {a} + {b}", target=target, top_all=ranked[0], second_all=ranked[1],
                top_excl=", ".join(excl[:3]), rank_excl=rank, cos_target=round(float(s[t]), 3),
                cos_top_all=round(float(s[order[0]]), 3))

an_glove = pd.DataFrame([analogy(lambda w: E[idx[w]], U, words, *q) for q in QUESTIONS])
an_glove.insert(0, "model", "GloVe")
print(an_glove.to_string())
t = E[idx["king"]] - E[idx["man"]] + E[idx["woman"]]
print("Euclidean distance of king - man + woman to king: %.2f, to queen: %.2f"
      % (np.linalg.norm(t - E[idx["king"]]), np.linalg.norm(t - E[idx["queen"]])))

## A dot-product probe: the plural direction
Plural direction = mean of (plural - singular) over 10 training pairs, scaled to length 1. Score of a word = its vector · that direction. Tested on 12 other pairs and on the number words.

In [ ]:
TRAIN = [("cat", "cats"), ("dog", "dogs"), ("car", "cars"), ("house", "houses"), ("tree", "trees"), ("book", "books"),
         ("bird", "birds"), ("river", "rivers"), ("city", "cities"), ("girl", "girls")]
TEST = [("apple", "apples"), ("horse", "horses"), ("door", "doors"), ("ship", "ships"), ("song", "songs"),
        ("table", "tables"), ("road", "roads"), ("flower", "flowers"), ("student", "students"),
        ("island", "islands"), ("window", "windows"), ("teacher", "teachers")]
NUMS = "one two three four five six seven eight nine ten".split()


def plural_probe(vec, model):
    p = np.mean([vec(b) - vec(a) for a, b in TRAIN], 0)
    p /= np.linalg.norm(p)
    rows = [(model, a, "singular", float(vec(a) @ p)) for a, b in TEST]
    rows += [(model, b, "plural", float(vec(b) @ p)) for a, b in TEST]
    rows += [(model, w, "number", float(vec(w) @ p)) for w in NUMS]
    d = pd.DataFrame(rows, columns=["model", "word", "kind", "score"])
    sing = d[d.kind == "singular"].score.values
    plur = d[d.kind == "plural"].score.values
    nums = d[d.kind == "number"].score
    rho = nums.rank().corr(pd.Series(range(10), index=nums.index).rank())
    print(f"{model}: plural scores higher in {(plur > sing).sum()} of {len(TEST)} held-out pairs; "
          f"mean singular {sing.mean():.2f}, mean plural {plur.mean():.2f}; number words {nums.round(2).tolist()}, "
          f"Spearman rank correlation with 1..10 = {rho:.2f}")
    return d

probe_glove = plural_probe(lambda w: E[idx[w]], "GloVe")

## The same inside a transformer: GPT-2 small's token-embedding table
Each word must be a single token with a leading space (" king"). GPT-2 uses the same table to turn its last vector back into token scores (tied unembedding).

In [ ]:
W = g.load_wte()
gvocab = [g.decode([i]) for i in range(len(W))]
Wu = W / np.linalg.norm(W, axis=1, keepdims=True)


def gvec(w):
    ids = g.encode(" " + w)
    assert len(ids) == 1, w
    return W[ids[0]]

print("embedding table:", W.shape)
M = W
common = np.array([g.encode(" " + w)[0] for w in words[:3000] if len(g.encode(" " + w)) == 1])
rows_gpt, base_gpt = pair_alignment(gvec, vocab_ids=common)
for r in rows_gpt:
    print(f"{r[0]:26s} {r[1]:.2f}")
print("random pairs: mean %.3f, 99th percentile of |cos| %.3f" % (base_gpt.mean(), np.percentile(abs(base_gpt), 99)))
an_gpt = pd.DataFrame([analogy(gvec, Wu, gvocab, *q, fmt=lambda w: " " + w) for q in QUESTIONS[:9]])
an_gpt.insert(0, "model", "GPT-2")
print(an_gpt.to_string())
probe_gpt = plural_probe(gvec, "GPT-2")

In [ ]:
pd.concat([an_glove, an_gpt]).to_csv(DATA / "analogies.csv", index=False)
pd.concat([probe_glove, probe_gpt]).round(4).to_csv(DATA / "plural_probe.csv", index=False)
al = pd.DataFrame([("GloVe", p, c) for p, c in rows_glove] + [("GPT-2", p, c) for p, c in rows_gpt],
                  columns=["model", "pair", "cosine"])
al.round(4).to_csv(DATA / "pair_alignment.csv", index=False)
pd.DataFrame({"model": ["GloVe"] * len(base_glove) + ["GPT-2"] * len(base_gpt),
              "cosine": np.r_[base_glove, base_gpt]}).round(4).to_csv(DATA / "random_pairs.csv", index=False)

## A 2-D plane for the animation
The plane through man with directions (woman - man) and (king - man) holds man, woman, king and king - man + woman exactly. Other words are projected onto it; `off_plane` is the distance they lose.

In [ ]:
o = E[idx["man"]]
e1 = E[idx["woman"]] - o
e1 /= np.linalg.norm(e1)
e2 = E[idx["king"]] - o - ((E[idx["king"]] - o) @ e1) * e1
e2 /= np.linalg.norm(e2)
rows = []
for name, v in [(w, E[idx[w]]) for w in ("man", "woman", "king", "queen")] + [("king − man + woman", t)]:
    d = v - o
    x, y = d @ e1, d @ e2
    rows.append((name, x, y, np.linalg.norm(d - x * e1 - y * e2)))
plane = pd.DataFrame(rows, columns=["word", "x", "y", "off_plane"]).round(3)
print(plane)
plane.to_csv(DATA / "analogy_plane.csv", index=False)